# 28 · Agentic RAG 与 Agentic Retrieval

> 把 RAG 从“固定流水线”升级为“**模型自主决策的循环**”：需要就检索、不够就再搜、错就纠正、够了才回答。

**本文件覆盖知识点**：Agentic RAG / Agentic Retrieval / Query·Search Planning / Tool Calling / Iterative Retrieval / Retrieval Decision / Stop Condition

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 从流水线到智能体

```text
普通 RAG:    Query → Retrieve → LLM           （一次到底）

Agentic RAG: User Query → Agent → Plan → Search
            → Evaluate → Search Again → Reason → Answer
                    ↑_________ 循环直到满意/达上限
```

智能体拥有三件套（01 课提过）：
- **Planning**：把任务拆步、决定检索还是直接答；
- **Tool Calling**：把“检索/查库/搜网页”做成工具，模型自主调用（Function Calling）；
- **Loop**：根据上一步结果决定继续还是停（Stop Condition）。

In [ ]:
# 真实的 Agentic 检索循环：每一步由 qwen-plus 决策「继续检索 / 直接回答」
# 与桩版的区别：检索走真混合召回(hybrid_retrieve)、决策是模型真输出的结构化 JSON、
# 答案是 qwen-plus 依据真实片段生成；决策里的 action 真的决定下一跳，不是写死的次数。
# 教学点不变：要不要再检索由模型自己判断，max_steps 只是兜底防死循环。
import json as _json

_AGENT_SYS = (
    '你是 Agentic RAG 的控制器。基于「用户问题」与「已收集的检索片段」决定下一步，只输出一个 JSON 对象：\n'
    '{"action": "retrieve", "query": "<改写后的检索词>", "reason": "<一句话理由>"}\n'
    '或 {"action": "answer", "reason": "<一句话理由>"}。\n'
    '已有片段足以回答问题时选 answer；信息不足或需要补充细节时选 retrieve，并给出更精准的检索词。'
    '禁止输出 JSON 以外的任何内容。')

class MiniAgentRAG:
    """迷你 Agent 化 RAG：模型每轮自主选 retrieve / answer，检索与生成都走真实底座"""

    def __init__(self, max_steps=3):
        self.max_steps = max_steps   # 兜底：最多检索几轮，防止模型不收敛
        self.collected = []          # 已收集的真实片段

    def _ctx(self):
        if not self.collected:
            return '（暂无，还没检索过）'
        return '\n'.join('[%d]（%s · %s）%s' % (i, d['source'], d['section'], d['text'][:200])
                         for i, d in enumerate(self.collected, 1))

    def decide(self, q):
        """真调 qwen-plus 输出结构化决策；返回的 JSON 直接驱动下面的控制流"""
        try:
            return chat_json('用户问题：%s\n\n已收集的检索片段：\n%s\n\n下一步？' % (q, self._ctx()),
                             system=_AGENT_SYS, temperature=0.1)
        except Exception as e:
            print('  决策 JSON 解析失败，按「信息已足够」收尾：', e)
            return None

    def retrieve(self, q, kw):
        """真跑一次混合检索（dense + BM25 + RRF 融合），返回并累积真实片段"""
        hits = hybrid_retrieve(kw, k=4)
        self.collected += hits
        return hits

    def answer(self, q):
        """用收集到的真实片段真调 qwen-plus 生成最终答案"""
        return chat('资料：\n%s\n\n问题：%s\n只依据资料作答，资料里没有的直说不知道。' % (self._ctx(), q))

    def run(self, q):
        for step in range(1, self.max_steps + 1):
            d = self.decide(q)
            print('step%d 决策 JSON：%s' % (step, _json.dumps(d, ensure_ascii=False)))
            if not d or d.get('action') == 'answer':
                print('  → 模型自评信息已足够，停止检索（Stop Condition：模型自主结束）')
                break
            kw = d.get('query') or q
            hits = self.retrieve(q, kw)
            print('  → 真实检索「%s」命中 %d 条：' % (kw, len(hits)))
            for h in hits:
                print('     [%s · %s] %s' % (h['source'], h['section'], h['text'][:44].replace('\n', ' ')))
        else:
            print('step%d 已达 max_steps=%d，强制收尾（Stop Condition：步数上限）' % (self.max_steps, self.max_steps))
        return self.answer(q)

_QUERY = '星云产品支持私有化部署吗？如果支持，最低硬件配置和年费大概是多少？'

if not _HAS_KEY:
    recorded(r"""用户问题： 星云产品支持私有化部署吗？如果支持，最低硬件配置和年费大概是多少？

step1 决策 JSON：{"action": "retrieve", "query": "星云产品私有化部署 支持情况 最低硬件配置 年费价格", "reason": "用户问题涉及三个关键信息点：是否支持私有化部署、最低硬件配置、年费金额，需一次性检索覆盖全部要点。"}
  → 真实检索「星云产品私有化部署 支持情况 最低硬件配置 年费价格」命中 4 条：
     [星云智能产品手册.md · 部署方式] ## 部署方式 产品支持公有云 SaaS 与私有化两种部署方式。公有云版本开通即可使用，
     [计费与SLA.md · 私有化部署计费] ## 私有化部署计费 私有化部署为一次性授权费 + 年度维保，授权费按坐席数与并发规模报
     [星云智能产品手册.md · 数据与安全] ## 数据与安全 - 公有云客户数据在传输与存储两侧均加密，不同租户之间逻辑隔离； -
     [星云客服FAQ.md · 部署相关] ## 部署相关 **Q：能私有化部署吗？** A：能。标准版及以上可申请私有化部署，需联
step2 决策 JSON：{"action": "answer", "reason": "已收集片段明确说明星云产品支持私有化部署（片段[1][2][4]），最低硬件配置为8核16线程/32GB内存/200GB SSD（片段[4]），年费为授权费的15%（片段[2]），虽未给出具体授权费数值，但问题中“大概”允许基于现有信息合理回答。"}
  → 模型自评信息已足够，停止检索（Stop Condition：模型自主结束）

最终答案（qwen-plus 依据 4 条真实片段生成）：
是的，星云产品支持私有化部署。依据资料[1][2][4]：

- **支持情况**：标准版及以上可申请私有化部署，需联系销售单独开通；企业版标配该能力（含一次现场实施）。
- **最低硬件配置**：依据资料[4]，最小要求为 **8 核 16 线程 / 32 GB 内存 / 200 GB SSD**，且需具备 Docker 环境；若知识库超过 100 万字，建议升级至 64 GB 内存。
- **年费**：资料中明确提到私有化部署计费模式为 **一次性授权费 + 年度维保费（授权费的 15%/年）**（见资料[2]），但**未提供授权费的具体金额或报价标准**，也未给出总年费的数值范围。

→ 因此，最低年费具体数额未知（资料中没有给出授权费基数，无法计算 15% 的维保费金额）。

→ 检索轮次是模型自己决定的（不是写死的固定次数），片段来源全部打印可核对。""",
             '录制于 2026-09-12，模型 qwen-plus（决策/生成）+ text-embedding-v3 / BM25（检索）')
else:
    _rag = MiniAgentRAG(max_steps=3)
    print('用户问题：', _QUERY)
    print()
    _ans = _rag.run(_QUERY)
    print()
    print('最终答案（qwen-plus 依据 %d 条真实片段生成）：' % len(_rag.collected))
    print(_ans)
    print()
    print('→ 检索轮次是模型自己决定的（不是写死的固定次数），片段来源全部打印可核对。')

In [ ]:
# 知识点·真调说明：Retrieval Decision / Tool 调用意图 —— 真调模型在“要不要检索、调哪个工具”上表态
import json as _json

_DECIDE_SYS = ('你是 Agentic RAG 的决策路由器。只输出一个 JSON 对象：'
               '若需检索知识库（涉及私有或未见过的资料）：'
               '{"action": "retrieve", "query": "<改写后的检索词>", "reason": "为什么检索"}；'
               '若常识可直接回答：'
               '{"action": "answer", "answer": "<直接回答>", "reason": "为什么不用检索"}。'
               '禁止输出其它文字。')

def _decide(qu):
    """真调 chat_json 拿结构化决策；解析失败返回 None，不让整格挂掉"""
    try:
        return chat_json('用户提问：' + qu, system=_DECIDE_SYS, temperature=0.1)
    except Exception as e:
        print('决策 JSON 解析失败：', e)
        return None

if not _HAS_KEY:
    recorded(r"""① 涉及私有产品资料 → 该走 retrieve
   决策 action=retrieve | 检索词：星云企业版私有化部署最低硬件要求 | 理由：该信息属于特定产品的私有部署规范，未在通用常识范围内，需查询官方文档或知识库确认具体参数

② 常识问题 → 该走 answer（Stop Condition：无需检索即可停）
   决策 action=answer | 理由：大语言模型是人工智能领域的通用基础概念，定义明确、广泛共识，无需依赖私有或未公开资料即可准确回答。

→ 同一个「决策接口」，不同 query 走出 retrieve / answer 两条路——这就是 Agentic RAG 的 Retrieval Decision 与 Stop Condition；上一格 MiniAgentRAG.decide() 用的就是这套真调用（不再是桩）。""", '录制于 2026-09-12，模型 qwen-plus')
else:
    print('① 涉及私有产品资料 → 该走 retrieve')
    _d1 = _decide('星云企业版私有化部署的最低硬件要求是多少？')
    if _d1:
        print('   决策 action=%s | 检索词：%s | 理由：%s'
              % (_d1.get('action'), _d1.get('query'), _d1.get('reason')))
    print()
    print('② 常识问题 → 该走 answer（Stop Condition：无需检索即可停）')
    _d2 = _decide('什么是大语言模型？')
    if _d2:
        print('   决策 action=%s | 理由：%s' % (_d2.get('action'), _d2.get('reason')))
    print()
    print('→ 同一个「决策接口」，不同 query 走出 retrieve / answer 两条路——'
          '这就是 Agentic RAG 的 Retrieval Decision 与 Stop Condition；'
          '上一格 MiniAgentRAG.decide() 用的就是这套真调用（不再是桩）。')

## 2. Agentic Retrieval 的关键设计点

| 设计 | 说明 |
|------|------|
| **Query/Search Planning** | 规划要搜什么、搜几次、每步换什么工具 |
| **Iterative Retrieval** | 多轮检索，每轮把“还缺什么”再搜一次 |
| **Self-Reflection / Correction** | 模型自评当前信息是否够、是否跑偏 |
| **Stop Condition** | 信息足够/步数上限/检索无增益 → 停 |
| **工具** | Tool Calling：检索器、SQL、网页搜索都是工具 |



In [ ]:
# 知识点·真调说明：Query / Search Planning —— 让模型把宽问题拆成“先搜什么、后搜什么”的检索计划
_q = '星云产品支持私有化部署吗？如果要私有化，官方给的最低硬件配置和年费大概是多少？'
print('宽问题：', _q)
print()
_llm_live(
    prompt=_q + '\n请为回答它规划检索步骤：需要搜几次、每步用什么关键词、查哪类资料。',
    system='你是 Agentic RAG 的查询规划器。先判断该问含哪几个子问题，再按依赖顺序给出检索计划。'
           '输出格式（不要多余解释，每行一条）：\n1) 子问题概述 —— 建议检索词\n2) …',
    fallback='未配置 Key 的固定样例：\n'
             '1) 星云是否支持私有化部署 —— 检索词：星云 私有化 部署方式\n'
             '2) 私有化最低硬件配置 —— 检索词：星云 私有化 硬件要求\n'
             '3) 私有化年费/授权 —— 检索词：星云 私有化 价格 授权',
    temperature=0.2,
)
print('→ 一条用户问题被拆成“分步检索计划”，Agent 按计划逐轮调用检索工具并自评“够不够”——这就是 Agentic Retrieval 的 Search Planning。')

## 3. 家族成员（下一课细看）

- **CRAG**：检索质量差就纠偏/联网补；
- **Self-RAG**：自己决定要不要检索、检索结果有没有用；
- **Adaptive RAG**：按问题类型路由到不同方案。

## 小结

- Agentic RAG = **决策循环**，检索成为模型可调用的“工具”；
- 控制好 **Stop Condition 与步数上限**是工程关键；
- 比固定流水线更鲁棒，但更贵、更难观测。